# PIGEON-AI: OSM destinations, transport and parks

Inspect the final OSM features and the approved nearest-park correction. Inputs: reconstructed OSM GeoPackage, authoritative features and validation reports. Outputs: tables and maps. Extraction and feature generation live in `src/pigeon_ai/osm.py`.

In [ ]:
from pathlib import Path
import sys

# Works when the kernel starts in the project root or notebooks/.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from pigeon_ai.paths import INTERIM, PROCESSED, CRS

assert (PROCESSED / "sydney_hex_features.gpkg").exists(), "Run python scripts/run_pipeline.py first."


## Destinations and context

Shops exclude only `shop=vacant`; food means cafe/restaurant. Bus stops use `highway=bus_stop`, stations use `railway=station`. Polygon objects become representative points in their original CRS before projection. Similar names are not deduplicated. Counts describe inside-cell placement and 250/500 m centre buffers; the contextual objects can lie outside the prediction boundary.

In [ ]:
features = gpd.read_file(PROCESSED / "sydney_hex_features.gpkg")
columns = [f"{kind}_{radius}m" for kind in ["shop", "food", "bus_stop", "train_station"] for radius in [250, 500]]
display(features[columns].describe())

## Park footprints and distance

Park area intersects a union of park polygons, preventing overlap double-counting. Area buffers retain the original Shapely circle approximation. Nearest distance uses a spatial index over **all available park polygons**, instead of the former 500 m subset. Distances refer to mapped park geometry, not an entrance or walking route. Coverage remains limited by OSM mapping and extraction.

In [ ]:
# Report from the earlier rewrite; only available if it was kept locally.
corrections_path = ROOT / "reports/validation/nearest_park_corrections.csv"
if corrections_path.exists():
    corrections = pd.read_csv(corrections_path)
    assert (corrections.change_m < 0).all()
    print(f"Corrected hexagons: {len(corrections):,}")
    display(corrections.nsmallest(10, "change_m"))
    print("Largest reduction (m):", -corrections.change_m.min())
else:
    print("Skipped: nearest-park corrections report not found.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
features.plot(ax=axes[0], column="park_area_500m_m2", cmap="Greens", legend=True, linewidth=0)
features.plot(ax=axes[1], column="distance_nearest_park_m", cmap="YlOrRd", legend=True, linewidth=0)
for ax, title in zip(axes, ["Mapped park footprint within 500 m (m²)", "Nearest available mapped park (m)"]):
    ax.set_title(title)
    ax.set_axis_off()
plt.tight_layout()
plt.show()